# Import Library

In [65]:
import pandas as pd
import numpy as np
from icdmappings import Mapper
from icdmappings import Validator

# Import Dataset

In [66]:
ed_df = pd.read_csv('data/diagnosis.csv')
ipd_df = pd.read_csv('data/diagnoses_icd.csv')

In [67]:
display(ed_df.info())
display(ipd_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 949172 entries, 0 to 949171
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   subject_id   949172 non-null  int64 
 1   stay_id      949172 non-null  int64 
 2   seq_num      949172 non-null  int64 
 3   icd_code     949172 non-null  object
 4   icd_version  949172 non-null  int64 
 5   icd_title    949172 non-null  object
dtypes: int64(4), object(2)
memory usage: 43.4+ MB


None

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4756326 entries, 0 to 4756325
Data columns (total 5 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   subject_id   int64 
 1   hadm_id      int64 
 2   seq_num      int64 
 3   icd_code     object
 4   icd_version  int64 
dtypes: int64(4), object(1)
memory usage: 181.4+ MB


None

# create Mapper, Validator Class

In [68]:
# Create Mapper, Validator Class
from icdmappings.mappers import ICD9toICD10
mapper = ICD9toICD10()
validator = Validator()

# Preprocessing

In [69]:
# Look at dataset
display(ed_df.head())
display(ed_df.shape)

display(ipd_df.head())
display(ipd_df.shape)

,subject_id,stay_id,seq_num,icd_code,icd_version,icd_title
0,15825222,39090953,1,486,9,"PNEUMONIA,ORGANISM UNSPECIFIED"
1,15825222,39090953,2,4254,9,PRIM CARDIOMYOPATHY NEC
2,11554870,37245764,1,5609,9,INTESTINAL OBSTRUCT NOS
3,19748558,30511202,1,49392,9,"ASTHMA, UNSPECIFIED, WITH ACUTE EXACERBATION"
4,18008347,34907903,1,7842,9,SWELLING IN HEAD & NECK


(949172, 6)

,subject_id,hadm_id,seq_num,icd_code,icd_version
0,10000032,22595853,1,5723,9
1,10000032,22595853,2,78959,9
2,10000032,22595853,3,5715,9
3,10000032,22595853,4,07070,9
4,10000032,22595853,5,496,9


(4756326, 5)

In [70]:
# Checking Missing Value
display(ed_df.isna().sum())
display(ipd_df.isna().sum())

subject_id     0
stay_id        0
seq_num        0
icd_code       0
icd_version    0
icd_title      0
dtype: int64

subject_id     0
hadm_id        0
seq_num        0
icd_code       0
icd_version    0
dtype: int64

In [71]:
# Check Duplicate 
print(ed_df.duplicated().sum())
print(ipd_df.duplicated().sum())

0
0


# Function Convert ICD code

In [72]:
def convert_icd9_to_icd10(data: pd.DataFrame):

    # Chceck Convert ED data ICD9 to 10 with Mapper
    data_copy = data.copy()   # Copy Dateset before input into mapper
    icd9_index = data_copy['icd_version'] == 9
    data_copy.loc[icd9_index, 'icd_code'] = mapper.map(data_copy.loc[icd9_index, 'icd_code'])

    # After Mapping ICD9 to ICD10, The invalids show 'NoDx' and None in 'icd_code'Column
    data_fail_idx = data_copy[data_copy['icd_code'] == 'NoDx'].index

    # records that ICD9 can't map to ICD 10
    data.loc[data_fail_idx, 'icd_version'] = 0 # Mark with icd_version = 0

    # Mapping
    icd9_index = data['icd_version'] == 9
    data.loc[icd9_index, 'icd_code'] = mapper.map(data.loc[icd9_index, 'icd_code'])

    return (data, data_fail_idx)

In [73]:
# Convert ED Dataset
ed_df, fail_index = convert_icd9_to_icd10(ed_df)

In [74]:
display(ed_df.loc[fail_index[0:5]]) # Show NoDX index
display(ed_df.head())
display(ed_df.shape)
display(ed_df['icd_version'].value_counts())
display(ed_df.isna().sum())

,subject_id,stay_id,seq_num,icd_code,icd_version,icd_title
141,15885421,32790351,3,E8704,0,ACC CUT/HEM W SCOPE EXAM
145,13758541,34384713,4,E9500,0,SUICIDE-ANALGESICS
219,12288821,32808916,4,E8538,0,ACC POISN-TRANQUILZR NEC
220,12288821,32808916,5,E8532,0,ACC POISN-BENZDIAZ TRANQ
372,12901637,37522731,2,E9270,0,OVEREXERTION FROM SUDDEN STRENUOUS MOVEMENT


,subject_id,stay_id,seq_num,icd_code,icd_version,icd_title
0,15825222,39090953,1,J189,9,"PNEUMONIA,ORGANISM UNSPECIFIED"
1,15825222,39090953,2,I428,9,PRIM CARDIOMYOPATHY NEC
2,11554870,37245764,1,K5660,9,INTESTINAL OBSTRUCT NOS
3,19748558,30511202,1,J45901,9,"ASTHMA, UNSPECIFIED, WITH ACUTE EXACERBATION"
4,18008347,34907903,1,R221,9,SWELLING IN HEAD & NECK


(949172, 6)

icd_version
10    481287
9     461381
0       6504
Name: count, dtype: int64

subject_id      0
stay_id         0
seq_num         0
icd_code       49
icd_version     0
icd_title       0
dtype: int64

In [75]:
# Convert IPD Dataset
ipd_df, fail_index = convert_icd9_to_icd10(ipd_df)

In [76]:
display(ipd_df.loc[fail_index[0:5]]) # Show NoDX index
display(ipd_df.head())
display(ipd_df.shape)
display(ipd_df['icd_version'].value_counts())
display(ipd_df.isna().sum())

,subject_id,hadm_id,seq_num,icd_code,icd_version
303,10000980,25242409,31,E9342,0
602,10001725,25563031,17,E9352,0
666,10001884,21268656,6,E9457,0
1214,10002155,28976727,12,E9331,0
1372,10002428,20321825,10,E9393,0


,subject_id,hadm_id,seq_num,icd_code,icd_version
0,10000032,22595853,1,K766,9
1,10000032,22595853,2,R188,9
2,10000032,22595853,3,K7469,9
3,10000032,22595853,4,B1920,9
4,10000032,22595853,5,J449,9


(4756326, 5)

icd_version
9     2730963
10    1989449
0       35914
Name: count, dtype: int64

subject_id         0
hadm_id            0
seq_num            0
icd_code       11865
icd_version        0
dtype: int64

# Add Column Chapter

In [77]:
from icdmappings.mappers import ICD10toChapters
mapper = ICD10toChapters()

def add_chapter_column(data: pd.DataFrame, mapper: Mapper) -> pd.DataFrame:
    data['chapter'] = mapper.map(data['icd_code'])
    return data

In [78]:
ed_df = add_chapter_column(ed_df,mapper)
ipd_df = add_chapter_column(ipd_df,mapper)

# Check Missing Chapter

In [79]:
display(ed_df.isna().sum())
display(ipd_df.isna().sum())

subject_id        0
stay_id           0
seq_num           0
icd_code         49
icd_version       0
icd_title         0
chapter        9578
dtype: int64

subject_id         0
hadm_id            0
seq_num            0
icd_code       11865
icd_version        0
chapter        56624
dtype: int64

# Drop Missing Value in Chapter or ICD code

In [80]:
# Drop Missing Value in Chapter
ed_df_copy = ed_df.dropna(subset=['icd_code', 'chapter'])
ipd_df_copy = ipd_df.dropna(subset=['icd_code', 'chapter'])

display(ed_df_copy.isna().sum())
display(ipd_df_copy.isna().sum())
display(ed_df_copy.shape)
display(ipd_df_copy.shape)

print('Missing Value in ED dataset', ed_df.shape[0]-ed_df_copy.shape[0], 'records')
print('Missing Value in IPD dataset', ipd_df.shape[0]-ipd_df_copy.shape[0], 'records')

subject_id     0
stay_id        0
seq_num        0
icd_code       0
icd_version    0
icd_title      0
chapter        0
dtype: int64

subject_id     0
hadm_id        0
seq_num        0
icd_code       0
icd_version    0
chapter        0
dtype: int64

(939594, 7)

(4699702, 6)

Missing Value in ED dataset 9578 records
Missing Value in IPD dataset 56624 records


### Drop process

In [81]:
ed_df = ed_df.dropna(subset=['icd_code', 'chapter'])
ipd_df= ipd_df.dropna(subset=['icd_code', 'chapter'])

In [82]:
display(ed_df.shape)
display(ipd_df.shape)

(939594, 7)

(4699702, 6)

### Change Data Type Chapter Column from String to Integer

In [83]:
ed_df['chapter'] = pd.to_numeric(ed_df['chapter'])
ipd_df['chapter'] = pd.to_numeric(ipd_df['chapter'])

# Transform Dataset into Frequency Table

In [84]:
def create_group_column_table(ed_data_df: pd.DataFrame, 
                              ipd_data_df: pd.DataFrame, 
                              group_column:str, 
                              count_column:str,
                              selected_group_column="None"
                              ) -> pd.DataFrame:
    
    if selected_group_column != "None":     # Case Create OverAll Table
        # Filter data by chapter
        ed_data_df_group_column = ed_data_df[ed_data_df[group_column] == selected_group_column]
        ipd_data_df_group_column = ipd_data_df[ipd_data_df[group_column] == selected_group_column]

        # Count occurrences of each ICD code
        ed_counts = ed_data_df_group_column[count_column].value_counts().rename('ED')
        ipd_counts = ipd_data_df_group_column[count_column].value_counts().rename('IPD')

    else:   # Case Create Chapter Table
        ed_counts = ed_data_df[count_column].value_counts().rename('ED')
        ipd_counts = ipd_data_df[count_column].value_counts().rename('IPD')

    # Combine the counts into a single DataFrame
    group_column_table_df = pd.concat([ed_counts, ipd_counts], axis=1).fillna(0).astype(int).reset_index()
    group_column_table_df.rename(columns={'index': count_column}, inplace=True)

    return group_column_table_df

# Create Frequency Table

In [85]:
table_overall = create_group_column_table(ed_df, ipd_df, 'chapter','chapter')
display(table_overall)

,chapter,ED,IPD
0,18,252201,370060
1,19,98659,151677
2,9,95028,732435
3,20,84982,162558
4,4,64489,556969
5,13,56951,191518
6,5,48131,322429
7,21,46463,731055
8,11,39539,315114
9,14,39200,227362


# Save Overall table

In [88]:
# Save Overall table
table_overall.to_csv('table/overall_tab.csv', header=True, index=False)

# Create Chapter Table and Save

In [92]:
# Only Chapter 1 - 21 (not include 22)
list_chapter = range(1,22)

for i in list_chapter:
    create_table = create_group_column_table(ed_df, ipd_df, 'chapter','icd_code', i)
    # display(test_create_table)

    create_table.to_csv(f'table/tab_chapter{i}.csv', header=True, index=False)